In [1]:
from transformers import BitsAndBytesConfig, LlavaNextForConditionalGeneration, AutoProcessor
from datasets import load_dataset
import torch
import json
from huggingface_hub import notebook_login
notebook_login()

In [2]:
MAX_LENGTH = 256
MODEL_ID = "llava-hf/llava-v1.6-mistral-7b-hf"
REPO_ID = "Farzad-R/llava-v1.6-mistral-7b-cordv2"

In [3]:
processor = AutoProcessor.from_pretrained(MODEL_ID)
processor.tokenizer.padding_side = "right" # during training, one always uses padding on the right

Some kwargs in processor config are unused and will not have any effect: vision_feature_select_strategy, image_token, num_additional_image_tokens, patch_size. 


In [4]:
from peft import LoraConfig, prepare_model_for_kbit_training, get_peft_model

USE_LORA = False
USE_QLORA = True

## Load model

# Three options for training, from the lowest precision training to the highest precision training:
# - QLora
# - Standard Lora
# - Full fine-tuning
if USE_QLORA or USE_LORA:
    if USE_QLORA:
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16
        )
    model = LlavaNextForConditionalGeneration.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.float16,
        quantization_config=bnb_config,
    )
else:
    # for full fine-tuning, we can speed up the model using Flash Attention
    # only available on certain devices, see https://github.com/Dao-AILab/flash-attention?tab=readme-ov-file#installation-and-features
    model = LlavaNextForConditionalGeneration.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.float16,
        _attn_implementation="flash_attention_2",
    )

config.json: 0.00B [00:00, ?B/s]

RuntimeError: No GPU found. A GPU is needed for quantization.